# 112 Python intermediate - Generatory i iteratory  v2.0

## Rozkład jazdy

1. 🔹 **Protokół iteratora** - `__iter__`, `__next__`, `StopIteration`
2. 🔹 **Funkcje generatorowe** - `yield`, lazy evaluation
3. 🔹 **`yield from` i generatory zaawansowane** - delegowanie, `send()`
4. 🔹 **Moduł `itertools`** - `chain`, `islice`, `groupby` i inne
   🐍 Ćwiczenia do każdej sekcji

---

## 1. 🔹 Protokół iteratora

Iterator (iterator) to obiekt, który produkuje kolejne wartości na
żądanie. Protokół iteratora składa się z dwóch metod:

- `__iter__(self)` - zwraca sam obiekt iteratora,
- `__next__(self)` - zwraca następną wartość lub rzuca `StopIteration`.

Iterowalny (iterable) to obiekt z metodą `__iter__()`, która zwraca
iterator. Każda pętla `for` wywołuje `iter()` na obiekcie, a następnie
wielokrotnie `next()` aż do `StopIteration`.

| Pojęcie | Metody | Przykłady |
|---|---|---|
| Iterable | `__iter__` | `list`, `str`, `dict`, nasze klasy |
| Iterator | `__iter__` + `__next__` | `iter(list)`, generator |

> 💡 Każdy iterator jest iterable (ma `__iter__`), ale nie odwrotnie.
> `list` jest iterable, ale nie jest iteratorem.

In [ ]:
class Countdown:
    """Iterator odliczający od n do 1."""

    def __init__(self, start: int):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self) -> int:
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


# pętla for używa protokołu iteratora automatycznie
for n in Countdown(5):
    print(n, end=' ')
print()

# to samo ręcznie
it = iter(Countdown(3))
print(next(it), next(it), next(it))

---

### 🐍 Ćwiczenia - protokół iteratora

**Ćw. 1.** Utwórz klasę `Range_` imitującą wbudowany `range(start, stop, step)`.
Klasa powinna działać w pętli `for` i obsługiwać `list(Range_(1, 10, 2))`.

**Ćw. 2.** Utwórz klasę `Fibonacci` - iterator zwracający kolejne
liczby Fibonacciego. Dodaj parametr `limit` (maksymalna wartość).

**Ćw. 3.** *(Trudniejsze)* Utwórz klasę `Cycle` - iterator, który
nieskończenie powtarza elementy z podanej sekwencji.
Pobierz pierwsze 10 elementów z `Cycle([1, 2, 3])` używając `next()`.
# hint: użyj operatora modulo do indeksowania

In [ ]:
# Ćw. 1
class Range_:
    def __init__(self, start: int, stop: int, step: int = 1):
        ...

    def __iter__(self):
        ...

    def __next__(self):
        ...


print(list(Range_(1, 10, 2)))   # [1, 3, 5, 7, 9]
print(list(Range_(0, 6)))       # [0, 1, 2, 3, 4, 5]

In [ ]:
# Ćw. 2
class Fibonacci:
    def __init__(self, limit: int):
        ...

    def __iter__(self):
        ...

    def __next__(self):
        ...


print(list(Fibonacci(100)))   # [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]

In [ ]:
# Ćw. 3
class Cycle:
    def __init__(self, sequence):
        ...

    def __iter__(self):
        ...

    def __next__(self):
        ...


c = Cycle([1, 2, 3])
result = [next(c) for _ in range(10)]
print(result)   # [1, 2, 3, 1, 2, 3, 1, 2, 3, 1]

---

## 2. 🔹 Funkcje generatorowe (`yield`)

Funkcja generatorowa (generator function) to funkcja zawierająca
słowo kluczowe `yield`. Wywołanie takiej funkcji zwraca obiekt
generatora - nie wykonuje kodu od razu.

Generatory są **leniwe** (lazy): obliczają kolejne wartości dopiero
wtedy, gdy są o to proszone. Dzięki temu zużywają minimalną pamięć
nawet przy pracy z milionami elementów.

```python
def gen():           # definicja
    yield 1          # pauzuje i zwraca 1
    yield 2          # pauzuje i zwraca 2
    yield 3          # pauzuje i zwraca 3
                     # StopIteration po powrocie z funkcji
```

> 💡 Generator expression `(x*2 for x in range(10))` to skrótowy
> zapis - tworzy generator bez definiowania funkcji.

In [ ]:
def squares(n: int):
    """Generator kwadratów liczb 1..n."""
    for i in range(1, n + 1):
        yield i * i


# generator jest iteratorem
gen = squares(5)
print(type(gen))
print(next(gen), next(gen))

# można go użyć w for
for sq in squares(5):
    print(sq, end=' ')
print()

# generator expression - bez def
even_squares = (x ** 2 for x in range(10) if x % 2 == 0)
print(list(even_squares))


def read_lines(text: str):
    """Leniwy parser wierszy - nie ładuje całości do pamięci."""
    for line in text.splitlines():
        stripped = line.strip()
        if stripped:
            yield stripped


sample = 'line1\n  line2  \n\nline3\n'
for line in read_lines(sample):
    print(repr(line))

---

### 🐍 Ćwiczenia - yield

**Ćw. 4.** Napisz generator `evens(n)` zwracający parzyste liczby
od 0 do n (włącznie). Wypisz je jako listę.

**Ćw. 5.** Napisz generator `running_average(numbers)`, który
przyjmuje iterable liczb i yieloduje bieżącą średnią po każdym
kolejnym elemencie.

**Ćw. 6.** *(Trudniejsze)* Napisz generator `flatten(nested)`,
który spłaszcza dowolnie zagnieżdżoną listę list do jednego
poziomu, np. `[1, [2, [3, 4]], 5]` → `1 2 3 4 5`.
# hint: sprawdź isinstance(item, list) i użyj rekurencji z yield from

In [ ]:
# Ćw. 4
def evens(n: int):
    ...


print(list(evens(10)))   # [0, 2, 4, 6, 8, 10]

In [ ]:
# Ćw. 5
def running_average(numbers):
    ...


print(list(running_average([1, 3, 5, 7])))   # [1.0, 2.0, 3.0, 4.0]

In [ ]:
# Ćw. 6
def flatten(nested):
    ...


print(list(flatten([1, [2, [3, 4]], 5])))   # [1, 2, 3, 4, 5]
print(list(flatten([[1, 2], [3, [4, [5]]]])))  # [1, 2, 3, 4, 5]

---

## 3. 🔹 `yield from` i generatory zaawansowane

`yield from iterable` to skrót dla pętli `for x in iterable: yield x`.
Pozwala delegować generację wartości do innego generatora lub dowolnego
iterable, co jest szczególnie przydatne przy rekurencji.

Generator może też **przyjmować wartości** przez metodę `send(value)`.
Wartość wysłana przez `send()` staje się wynikiem wyrażenia `yield`
w środku generatora.

| Metoda | Opis |
|---|---|
| `next(gen)` | zwraca następną wartość |
| `gen.send(val)` | wysyła wartość do generatora i zwraca następny `yield` |
| `gen.throw(exc)` | rzuca wyjątek wewnątrz generatora |
| `gen.close()` | kończy generator (rzuca `GeneratorExit`) |

In [ ]:
# yield from - delegowanie
def chain_(*iterables):
    """Łączy kilka iterable w jeden ciąg."""
    for it in iterables:
        yield from it


print(list(chain_([1, 2], [3, 4], [5])))  # [1, 2, 3, 4, 5]


# send() - generator jako koproces
def accumulator():
    """Przyjmuje liczby przez send() i yieloduje bieżącą sumę."""
    total = 0
    while True:
        value = yield total   # pauzuje, zwraca total, przyjmuje value
        if value is None:
            break
        total += value


acc = accumulator()
next(acc)          # inicjalizacja (do pierwszego yield)
print(acc.send(10))  # 10
print(acc.send(5))   # 15
print(acc.send(3))   # 18

---

### 🐍 Ćwiczenia - yield from i send

**Ćw. 7.** Napisz generator `tree_values(node)`, który przechodzi
drzewo słownikowe `{'value': 1, 'children': [...]}` w kolejności
pre-order i yieloduje wartości węzłów. Użyj `yield from`.

**Ćw. 8.** Napisz generator `interleave(*iters)`, który naprzemiennie
yieloduje elementy z wielu iterable (jak splot), np.
`interleave([1,2,3], ['a','b','c'])` → `1 a 2 b 3 c`.

**Ćw. 9.** *(Trudniejsze)* Napisz generator `moving_window(iterable,
size)` zwracający krotki kolejnych `size` elementów - okno przesuwne.
`moving_window([1,2,3,4,5], 3)` → `(1,2,3) (2,3,4) (3,4,5)`.
# hint: użyj collections.deque z maxlen=size

In [ ]:
# Ćw. 7
def tree_values(node: dict):
    ...


tree = {
    'value': 1,
    'children': [
        {'value': 2, 'children': []},
        {'value': 3, 'children': [
            {'value': 4, 'children': []}
        ]},
    ],
}
print(list(tree_values(tree)))   # [1, 2, 3, 4]

In [ ]:
# Ćw. 8
def interleave(*iters):
    ...


print(list(interleave([1, 2, 3], ['a', 'b', 'c'])))
# [1, 'a', 2, 'b', 3, 'c']

In [ ]:
# Ćw. 9
from collections import deque

def moving_window(iterable, size: int):
    ...


print(list(moving_window([1, 2, 3, 4, 5], 3)))
# [(1, 2, 3), (2, 3, 4), (3, 4, 5)]

---

## 4. 🔹 Moduł `itertools`

Moduł `itertools` dostarcza wydajne iteratory do typowych operacji.
Wszystkie działają leniwie - nie tworzą list w pamięci.

| Funkcja | Opis | Przykład |
|---|---|---|
| `chain(*iters)` | łączy iterable | `chain([1,2],[3,4])` → `1 2 3 4` |
| `islice(it,n)` | pierwsze n elementów | `islice(count(),5)` → `0..4` |
| `count(start,step)` | nieskończony licznik | `count(0,2)` → `0 2 4..` |
| `cycle(it)` | nieskończone powtarzanie | `cycle([1,2])` → `1 2 1 2..` |
| `repeat(val,n)` | powtarza wartość n razy | `repeat(0,3)` → `0 0 0` |
| `groupby(it,key)` | grupuje kolejne elementy | wg klucza |
| `product(*iters)` | iloczyn kartezjański | `product([1,2],[a,b])` |
| `combinations(it,r)` | kombinacje | `comb([1,2,3],2)` |
| `permutations(it,r)` | permutacje | `perm([1,2,3],2)` |
| `compress(data,sel)` | filtruje wg maski | `compress(abc,101)` → `a c` |

> 💡 `itertools.chain.from_iterable(iters)` to wygodny odpowiednik
> `chain(*iters)`, gdy mamy iterable iterabli.

In [ ]:
import itertools

# chain - łączenie
print(list(itertools.chain([1, 2], [3, 4], [5])))

# islice - pierwsze n (z nieskończonego iteratora)
print(list(itertools.islice(itertools.count(0, 5), 6)))

# groupby - grupowanie kolejnych równych elementów
data = [('a', 1), ('a', 2), ('b', 3), ('b', 4), ('a', 5)]
for key, group in itertools.groupby(data, key=lambda x: x[0]):
    print(key, list(group))

# combinations i product
print(list(itertools.combinations([1, 2, 3], 2)))
print(list(itertools.product([0, 1], repeat=3)))

---

### 🐍 Ćwiczenia - itertools

**Ćw. 10.** Użyj `itertools.chain.from_iterable`, aby spłaszczyć
listę list: `[[1,2],[3],[4,5,6]]` → `[1,2,3,4,5,6]`.
Następnie użyj `itertools.compress`, aby wybrać co drugi element.

**Ćw. 11.** Wygeneruj wszystkie 3-literowe kombinacje z liter
`['a','b','c','d']` bez powtórzeń. Ile jest takich kombinacji?
Następnie wygeneruj permutacje 2 z 4 liter.

**Ćw. 12.** *(Trudniejsze)* Masz listę transakcji:
`transactions = [('Alice',100),('Bob',50),('Alice',200),('Bob',75)]`.
Użyj `sorted` + `itertools.groupby`, aby zsumować wartości
transakcji dla każdej osoby.
# hint: groupby działa tylko na posortowanych danych

In [ ]:
# Ćw. 10
import itertools

nested = [[1, 2], [3], [4, 5, 6]]
flat = ...
print(flat)

# co drugi element (indeksy 0, 2, 4, ...)
mask = ...
print(list(itertools.compress(flat, mask)))

In [ ]:
# Ćw. 11
import itertools

letters = ['a', 'b', 'c', 'd']
combos = ...
print(combos)
print(f'Combinations: {len(combos)}')

perms = ...
print(perms)

In [ ]:
# Ćw. 12
import itertools

transactions = [
    ('Alice', 100), ('Bob', 50),
    ('Alice', 200), ('Bob', 75),
]
# hint: posortuj wg nazwy, potem groupby
totals = ...
print(totals)  # {'Alice': 300, 'Bob': 125}